# 01 — Schema Exploration

Inspect the processed RCSB `.npz` files to confirm what data fields are available.
This is the prerequisite check before writing any pipeline code.

**Key questions:**
1. Are water atom records present? What residue names / chain types?
2. Are B-factors available?
3. Are occupancies and alt-loc identifiers available?
4. Is unit cell / space group information available?

In [ ]:
import sys
import numpy as np
import json

sys.path.insert(0, '../scripts')
from npz_io import (
    load_npz, get_water_coords, get_water_bfactors,
    count_waters, get_first_valid_polymer_chain_idx, get_polymer_ca_coords,
    STRUCTURES_DIR, DATA_DIR
)

print('Data directory:', DATA_DIR)
print('Structures directory:', STRUCTURES_DIR)

## 1. Raw npz arrays for a representative structure (101m — myoglobin)

In [ ]:
d = load_npz('101m')
print('Keys:', list(d.keys()))
for k in d.keys():
    v = d[k]
    print(f'\n{k}: shape={v.shape}, dtype={v.dtype}')
    if v.dtype.names:
        print(f'  fields: {v.dtype.names}')

## 2. Chain types and water chains

In [ ]:
chains = d['chains']
print('mol_type legend: 0=polymer, 3=ligand, 4=water')
print()
for c in chains:
    print(f"  name={c['name']!r:10s}  mol_type={c['mol_type']}  res_num={c['res_num']}")

## 3. Residue names — confirm HOH present

In [ ]:
residues = d['residues']
names = residues['name']
from collections import Counter
print('Residue name counts (top 30):')
for name, count in Counter(names.tolist()).most_common(30):
    print(f'  {name:6s}: {count}')

water_mask = (names == 'HOH') | (names == 'WAT')
print(f'\nWater residues (HOH/WAT): {water_mask.sum()}')

## 4. Atom fields — B-factor, occupancy, alt-loc availability

In [ ]:
atoms = d['atoms']
print('Atom dtype fields:', atoms.dtype.names)
print()
print('Field availability check:')
print(f"  bfactor:   {'YES' if 'bfactor' in atoms.dtype.names else 'MISSING'}")
print(f"  occupancy: {'YES' if 'occupancy' in atoms.dtype.names else 'MISSING'}")
print(f"  alt_loc:   {'YES' if 'alt_loc' in atoms.dtype.names or 'altloc' in atoms.dtype.names else 'MISSING'}")
print()
print('Sample atoms (first 3):')
for a in atoms[:3]:
    print(' ', a)

## 5. Water atom details — coordinates and B-factors

In [ ]:
water_coords = get_water_coords(d)
water_bfactors = get_water_bfactors(d)
print(f'Water oxygen coords shape: {water_coords.shape}')
print(f'Water B-factor range: {water_bfactors.min():.1f} – {water_bfactors.max():.1f} Å²')
print(f'Water B-factor median: {np.median(water_bfactors):.1f} Å²')
print()
print('First 5 water coordinates:')
for i, (c, b) in enumerate(zip(water_coords[:5], water_bfactors[:5])):
    print(f'  {i}: xyz={c}  B={b:.1f}')

## 6. JSON record — unit cell / crystal symmetry check

In [ ]:
rec = json.load(open(DATA_DIR / 'records' / '101m.json'))
print('JSON record fields:')
print(json.dumps(rec, indent=2))
print()
crystal_keys = [k for k in rec.keys() if any(x in k.lower() for x in ['cell', 'space', 'cryst', 'symmetry'])]
print('Crystal/symmetry keys in record:', crystal_keys if crystal_keys else 'NONE FOUND')

## 7. Survey across a sample of structures

In [ ]:
sample_pdbs = ['101m', '1thm', '1a4w', '4hhb', '3bwm', '1l2y', '2hhb', '1hho', '3ca2', '2lyz']
print(f"{'PDB':6s}  {'#waters':8s}  {'B-factor range':20s}  {'CA chain 0':12s}")
print('-' * 55)
for pdb in sample_pdbs:
    try:
        npz = load_npz(pdb)
        wc = count_waters(npz)
        bf = get_water_bfactors(npz)
        ci = get_first_valid_polymer_chain_idx(npz)
        if ci is not None and wc > 0:
            ca = get_polymer_ca_coords(npz, ci)
            bf_str = f'{bf.min():.0f}-{bf.max():.0f}' if len(bf) > 0 else 'n/a'
            print(f"{pdb:6s}  {wc:8d}  {bf_str:20s}  {ca.shape[0]} residues")
        else:
            print(f"{pdb:6s}  {wc:8d}  {'no waters or polymer':20s}")
    except FileNotFoundError:
        print(f"{pdb:6s}  not in dataset")

## Summary of schema findings

| Field | Available? | Notes |
|-------|-----------|-------|
| Water oxygen coordinates | ✅ | `mol_type=4` chains, residue name HOH, atom name O |
| B-factors | ✅ | `atoms['bfactor']` — all atoms including waters |
| Occupancy | ❌ | Not stored in npz |
| Alt-loc identifiers | ❌ | Not stored in npz |
| Unit cell / space group | ❌ | Not in npz or JSON records |

**Implications:**
- We can use B-factors for water quality filtering (Carugo mobility approach viable).
- Occupancy filtering is impossible from these files — if alternate conformations were present in the original PDB, they may have been collapsed during preprocessing.
- Crystal-contact filtering (step 7 in the pipeline) requires loading original PDB files or CIF files from the source PDB mirror, not from the npz preprocessed files.